<a href="https://colab.research.google.com/github/oangueram/Lloguer-Catalunya-ML/blob/main/Analisi_Lloguer_Catalunya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anàlisi i Predicció del Mercat de Lloguer a Catalunya (2007-2025)
**Autor:** Oriol Anguera Milà

**Eines:** Python (Pandas, Plotly, Scikit-Learn, Numpy), GeoJSON cartogràfic.

**Objectiu del Projecte:**
Aquest projecte estudia l'evolució del preu del lloguer als municipis catalans des del 2007 fins a l'actualitat per extreure'n conclusions de negoci. A través de tècniques d'anàlisi de dades i Machine Learning, l'objectiu és identificar oportunitats d'inversió, predir la tendència econòmica de grans capitals (Barcelona) i detectar automàticament "Cignes Negres" (bombolles o mercats d'alt risc) mitjançant algorismes no supervisats.

In [ ]:
import pandas as pd
import json
import requests
import zipfile
import io
import plotly.express as px
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import silhouette_score
from sklearn.model_selection import TimeSeriesSplit


In [ ]:
ruta = 'https://raw.githubusercontent.com/oangueram/Lloguer-Catalunya-ML/main/ambits_trimestral_lloguer.csv'
df = pd.read_csv(ruta)
df.head()

# 1. Data Cleaning & Wrangling (Preparació de Dades)

In [ ]:
df_net = df.dropna().copy()

df_net['Renda'] = (
    df_net['Renda']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# Filtrem outliers agrupant per any i localitat per evitar l'efecte de la
# inflació i la desigualtat de preus entre diferents territoris
Q1 = df_net.groupby(['Any', 'Nom territori'])['Renda'].transform(
    'quantile', 0.25
)

Q3 = df_net.groupby(['Any', 'Nom territori'])['Renda'].transform(
    'quantile', 0.75
)

IQR = Q3 - Q1
limit_inferior = Q1 - 1.5 * IQR
limit_superior = Q3 + 1.5 * IQR

df_net = df_net[
    (df_net['Renda'] >= limit_inferior) & (df_net['Renda'] <= limit_superior)
]

# 2. Exploratory Data Analysis - EDA (Anàlisi Exploratòria)

Analitzarem la distribució general dels preus i n'extreurem els rànquings de preu, creixement i volatilitat. També relacionarem preu, creixement i volum de mercat en un mateix gràfic.

In [ ]:
fig_hist = px.histogram(
    df_net[df_net['Any'] == 2025],
    x='Renda',
    nbins=30,
    title=f"Distribució del preu del lloguer a Catalunya (2025)",
    color_discrete_sequence=['#636EFA']
)
fig_hist.show()

In [ ]:
evolucio_preus = (
    df_net.groupby(['Any', 'Nom territori'])['Renda']
    .mean()
    .reset_index()
)

top_cars = evolucio_preus.sort_values(by='Renda', ascending=False)
top_cars.head(10)

In [ ]:
taula_anys = evolucio_preus.pivot(
    index='Nom territori', columns='Any', values='Renda'
)

anys_disponibles = sorted(evolucio_preus['Any'].unique())
any_actual = anys_disponibles[-1]
any_anterior = anys_disponibles[-2]

taula_anys['Creixement (%)'] = (
    (taula_anys[any_actual] - taula_anys[any_anterior])
    / taula_anys[any_anterior]
) * 100

top_creixement = taula_anys.sort_values(by='Creixement (%)', ascending=False)
top_3_pujades = top_creixement.dropna(subset=['Creixement (%)']).head(3)

print(f"Màxims creixements entre {any_anterior} i {any_actual}:")
top_3_pujades.columns.name = None
top_3_pujades[['Creixement (%)']]

In [ ]:
any_base = anys_disponibles[0]
any_actual = anys_disponibles[-1]

taula_anys['Creixement Històric (%)'] = (
    (taula_anys[any_actual] - taula_anys[any_base])
    / taula_anys[any_base]
) * 100

top_historic = taula_anys.sort_values(
    by='Creixement Històric (%)', ascending=False
)

top_3_historic = top_historic.dropna(
    subset=['Creixement Històric (%)']
).head(3)

print(f"Màxims creixements entre {any_base} i {any_actual}:")
top_3_historic.columns.name = None
top_3_historic[['Creixement Històric (%)']]

In [ ]:
volatilitat = (
    df_net.groupby('Nom territori')['Renda']
    .std()
    .reset_index()
    .rename(columns={'Renda': 'Volatilitat_Std'})
)

top_volatils = volatilitat.sort_values(
    by='Volatilitat_Std', ascending=False
)

print('Més volàtils')
top_volatils.head(5)


# 3. Geospatial Analysis (Mapa Interactiu de Catalunya)

Mapegem les dades per entendre l'efecte "taca d'oli" de Barcelona i localitzar visualment les zones de tensió de preus a Catalunya.

Les àrees no ombrejades del mapa corresponen a municipis on no es disposa de dades de les rendes.

In [ ]:
ruta_zip_github = 'https://raw.githubusercontent.com/oangueram/Lloguer-Catalunya-ML/main/municipis.zip'
resposta = requests.get(ruta_zip_github)
fitxer_zip = zipfile.ZipFile(io.BytesIO(resposta.content))

amb_geojson_obert = fitxer_zip.open('municipis.geojson')
geojson_cat = json.load(amb_geojson_obert)

In [ ]:
dades_mapa = (
    df_net[df_net['Any'] == any_actual]
    .groupby(['Codi territorial', 'Nom territori'])
    .agg({'Renda': 'mean'})
    .reset_index()
    .rename(columns={'Renda': 'Preu_Mitja'})
)

dades_mapa['Codi_5'] = dades_mapa['Codi territorial'].astype(str).str.zfill(5)

for feature in geojson_cat['features']:
    feature['properties']['codi_5_mapa'] = (
        feature['properties']['codimuni'][:5]
    )

mapa = px.choropleth_mapbox(
    dades_mapa,
    geojson=geojson_cat,
    locations='Codi_5',
    featureidkey='properties.codi_5_mapa',
    color='Preu_Mitja',
    hover_name='Nom territori',
    color_continuous_scale="Reds",
    mapbox_style="carto-positron",
    zoom=6.5,
    center={"lat": 41.7, "lon": 1.5},
    opacity=0.7,
    title="Mapa de Preus del Lloguer a Catalunya"
)

mapa.show()

# 4. Predicció de Mercat (Machine Learning Supervisat)

Agafem perspectiva i ens mirem l'evolució dels preus històrics per poder predir els preus futurs. Comencem estudiant la solidesa dels models polinòmics.

In [ ]:
dades_bcn = evolucio_preus[
    evolucio_preus['Nom territori'] == "Barcelona"
].dropna()

X = dades_bcn[['Any']].values
y = dades_bcn['Renda'].values
graus = [0, 1, 2, 3, 4, 5]
resultats_mae = []

# Els partim en 3 per predir 3 grups de dades i fer la mitjana de l'error.
# Evitem que el model "encerti de casualitat"
tscv = TimeSeriesSplit(n_splits=3)

for grau in graus:
    errors_fold = []

    for train_index, test_index in tscv.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        model_prova = make_pipeline(
            PolynomialFeatures(degree=grau), LinearRegression()
        )
        model_prova.fit(X_train, y_train)
        prediccions = model_prova.predict(X_test)

        error_fold = mean_absolute_error(y_test, prediccions)
        errors_fold.append(error_fold)

    error_mae_mitja = np.mean(errors_fold)
    resultats_mae.append(error_mae_mitja)

    print(f"Error mitjà MAE (Grau {grau}): {round(error_mae_mitja, 2)}€")

La regressió lineal (grau 1) i el model de grau 2 demostren ser els models polinòmics més robustos ja que minimitzen l'error. En voler extrapolar amb un model polinòmic de grau major que 2 observem que pateixen d'un gran nivell d'overfitting. Per el model quadràtic, creixerà a l'infinit més ràpidament que el lineal, de manera que tot i que l'error del lineal sigui lleugerament major que el del quadràtic, ens quedaríem amb el lineal.

In [ ]:
model = LinearRegression()
model.fit(X, y)
r2 = model.score(X, y)

anys_totals = np.arange(X.min(), 2030).reshape(-1, 1)
prediccio_continua = model.predict(anys_totals)

grafic_pred = go.Figure()

grafic_pred.add_trace(
    go.Scatter(
        x=dades_bcn['Any'],
        y=y,
        mode='markers',
        name='Dades Reals',
        marker=dict(size=10, color='blue')
    )
)

grafic_pred.add_trace(
    go.Scatter(
        x=anys_totals.flatten(),
        y=prediccio_continua,
        mode='lines',
        name=f'Regressio Lineal(R2={round(r2, 2)})',
        line=dict(dash='dot', color='red', width=3)
    )
)

grafic_pred.update_layout(
    title='Regressió Lineal del preu del lloguer a Barcelona',
    xaxis_title='Any',
    yaxis_title='Preu Mitja Lloguer'
)

grafic_pred.show()

Tot i que el model lineal estableix una bona tendència base, té molt marge de millora:, ja que el mercat immobiliari es mou en cicles que no podem captar amb una recta. Per solucionar-ho, aplicarem "Feature Engineering" incorporant variables de retard (Lag Features). Així, transformem l'algorisme en un model autoregressiu, permetent-li avaluar el preu i la tendència del passat per predir el futur amb molta més precisió.

In [ ]:
dades_bcn = dades_bcn.sort_values(by='Any')
dades_bcn['Preu_Lag_1'] = dades_bcn['Renda'].shift(1)
dades_bcn['Preu_Lag_2'] = dades_bcn['Renda'].shift(2)

dades_bcn['Tendencia_Previa'] = (
    dades_bcn['Preu_Lag_1'] - dades_bcn['Preu_Lag_2']
)

dades_bcn_model = dades_bcn.dropna()

X = dades_bcn_model[['Any', 'Preu_Lag_1', 'Tendencia_Previa']].values
y = dades_bcn_model['Renda'].values

tscv = TimeSeriesSplit(n_splits=3)
errors_mae = []
r2_scores = []

for train_index, test_index in tscv.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model_cv = LinearRegression()
    model_cv.fit(X_train, y_train)
    pred_test = model_cv.predict(X_test)

    errors_mae.append(mean_absolute_error(y_test, pred_test))
    r2_scores.append(r2_score(y_test, pred_test))

error_mae_mitja_cv = np.mean(errors_mae)

model_final = LinearRegression()
model_final.fit(X, y)
prediccio_historica = model_final.predict(X)

anys_futurs = np.arange(dades_bcn_model['Any'].max() + 1, 2030)
prediccions_futures = []

preu_t1 = y[-1]
preu_t2 = y[-2]

for any_f in anys_futurs:
    tendencia = preu_t1 - preu_t2
    X_futur = np.array([[any_f, preu_t1, tendencia]])
    preu_nou = model_final.predict(X_futur)[0]

    prediccions_futures.append(preu_nou)
    preu_t2 = preu_t1
    preu_t1 = preu_nou

anys_plot = np.concatenate((dades_bcn_model['Any'].values, anys_futurs))
prediccions_plot = np.concatenate(
    (prediccio_historica, prediccions_futures)
)

grafic_pred = go.Figure()

grafic_pred.add_trace(
    go.Scatter(
        x=dades_bcn['Any'],
        y=dades_bcn['Renda'],
        mode='markers',
        name='Dades Reals',
        marker=dict(size=10, color='blue')
    )
)

grafic_pred.add_trace(
    go.Scatter(
        x=anys_plot,
        y=prediccions_plot,
        mode='lines',
        name=f'Regressió (amb Lags)',
        line=dict(dash='dot', color='red', width=3)
    )
)

grafic_pred.update_layout(
    title='Regressió Lineal (amb Lags) del preu del lloguer a Barcelona',
    xaxis_title='Any',
    yaxis_title='Preu Mitjà Lloguer'
)

grafic_pred.show()
print(f"Error mitjà (MAE) real: {round(error_mae_mitja_cv, 2)}€")

Observem que amb el nou model hem reduit bastant l'error mitjà. Per tant, tenim un model millor. Això sí, no ens servirà per fer prediccions a llarg termini perque utilitza les prediccions anteriors per fer les futures, fet que fara que els errors s'acumulin exponencialment.

# 5. Segmentació i Detecció d'Anomalies (Machine Learning No Supervisat)


Busquem trobar els millors municipis on invertir. Per fer la tria, ens basarem en l'algorisme K-means i en els següents paràmetres:
1. Preu Mitjà Actual: defineix la barrera d'entrada.
2. Creixement Històric (%): mesura la inèrcia de revalorització de la zona.
3. Total Contractes / Volum: actua com a indicador de liquiditat del mercat i d'estabilitat davant valors extrems.



Primer, però, observem que els parametres que emprarem per a l'algorisme no tenen correlació greu entre ells. Una correlació forta entre paràmetres (>0.7) seria semblant a duplicar el pes d'un d'ells, cosa que afectaria al model. Veiem que no tenen correlació forta a la següent matriu:

In [ ]:
dades_actuals = (
    df_net[df_net['Any'] == any_actual]
    .groupby('Nom territori')
    .agg({'Renda': 'mean', 'Habitatges': 'sum'})
    .reset_index()
    .rename(
        columns={
            'Renda': 'Preu_Mitja_Actual',
            'Habitatges': 'Total_Contractes'
        }
    )
)

dades_cluster = pd.merge(
    dades_actuals,
    taula_anys[['Creixement Històric (%)']],
    left_on='Nom territori',
    right_index=True
).dropna()

columnes_model = [
    'Preu_Mitja_Actual',
    'Creixement Històric (%)',
    'Total_Contractes'
]

matriu_corr = dades_cluster[columnes_model].corr()

fig_corr = px.imshow(
    matriu_corr,
    text_auto=True,
    aspect="auto",
    title="Matriu de Correlació (Features del Model)",
    color_continuous_scale='RdBu_r'
)

fig_corr.show()

Emprant l'algorisme K-Means segmentarem els municipis en k grups de localitats amb paràmetres semblants. Intentarem escollir un grup que maximitzi el Creixement Històric i el Volum i minimitzi el Preu Mitjà Actual.


Per a la segmentació, calcularem el nombre de clústers (grups) idonis emprant el Mètode del Colze i el Coeficient de Silhouette.

In [ ]:
wcss = []
silhouette = []
scaler = StandardScaler()

variables_matematiques = scaler.fit_transform(
    dades_cluster[
        ['Preu_Mitja_Actual', 'Creixement Històric (%)', 'Total_Contractes']
    ]
)

model_kmeans = KMeans(n_clusters=4, random_state=42)
k_valors = range(2, 10)

for k in k_valors:
    model_tmp = KMeans(n_clusters=k, random_state=42)
    etiquetes = model_tmp.fit_predict(variables_matematiques)
    wcss.append(model_tmp.inertia_)
    silhouette.append(silhouette_score(variables_matematiques, etiquetes))

grafic_justificacio = sp.make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Mètode del Colze (WCSS)', 'Coeficient de Silhouette')
)

grafic_justificacio.add_trace(
    go.Scatter(
        x=list(k_valors),
        y=wcss,
        mode='lines+markers',
        name='WCSS'
    ),
    row=1,
    col=1
)

grafic_justificacio.add_trace(
    go.Scatter(
        x=list(k_valors),
        y=silhouette,
        mode='lines+markers',
        name='Silhouette',
        marker=dict(color='green')
    ),
    row=1,
    col=2
)

grafic_justificacio.update_layout(
    title_text='Justificació Matemàtica del nombre de clústers (k)',
    showlegend=False
)

grafic_justificacio.show()

El Coeficient de Silhouette mostra el seu màxim a k=2. No seria una bona estratègia dividir les dades en dos grups ja que no ens aportaria informació de valor. Observem que, en el gràfic del mètode del colze, el mòdul del pendent decau dràsticament a partir de k=4, és per això que dividirem les dades en 4 segments.
Aquesta decisió es veu recolzada pel fet que permet alinear els clústers amb els 4 quadrants financers clàssics (Premium, Emergent, Estancat, Estable), fent un gran guany estratègic.


In [ ]:
dades_cluster['Segment_Mercat'] = model_kmeans.fit_predict(
    variables_matematiques
)

dades_cluster['Segment_Mercat'] = dades_cluster['Segment_Mercat'].astype(str)

grafic_segments = px.scatter(
    dades_cluster,
    x='Preu_Mitja_Actual',
    y='Creixement Històric (%)',
    color='Segment_Mercat',
    size='Total_Contractes',
    hover_name='Nom territori',
    title='Segmentacio de Mercat amb IA (Clustering k-Means)'
)

grafic_segments.show()

Ens adonem que el grup numero 0 (blau) és el mercat estancat. El número 1 (lila) és el mercat estable. El número 2 (taronja) fa referència a municipis amb preus de lloguer baixos pero rendibilitat alta, és a dir, als mercats emergents. Finalment, el 3 (verd) és el mercat prèmium.

El segment que millor s'adapta al que busquem es el 2. Té preus baixos, rendibilitats altes. Cal observar, però, que paguem el preu de tenir un volum de mercat petit.
Els municipis que hi pertanyen són:

In [ ]:
oportunitats_inversio = dades_cluster[dades_cluster['Segment_Mercat'] == '2']

columnes_inversio = [
    'Nom territori',
    'Preu_Mitja_Actual',
    'Creixement Històric (%)',
    'Total_Contractes'
]

llista_oportunitats = oportunitats_inversio[columnes_inversio]

llista_oportunitats = llista_oportunitats.sort_values(
    by='Creixement Històric (%)', ascending=False
)

llista_oportunitats = llista_oportunitats.reset_index(drop=True)

llista_oportunitats

Per compensar el moderat volum de mercat del grup 2, ara aplicarem un Isolation Forest per detectar anomalies i descartar els municipis amb més risc. Estudiarem 3 paràmetres (preu, creixement, volatilitat) i aïllarem els municipis amb un comportament més extrem, als quals no seria recomanable invertir. Nosaltres aïllarem els municipis dins del 5% més anòmal.

Hem establert el paràmetre de contaminació de l'Isolation Forest al 5% com un criteri de gestió de cartera. El valor podria canviar depenent de la nostra aversió al risc o la volatilitat del mercat.

In [ ]:
dades_3d = pd.merge(
    dades_cluster, volatilitat, on='Nom territori'
).dropna()

model_iso = IsolationForest(contamination=0.05, random_state=42)

dades_3d['Es_Anomalia'] = model_iso.fit_predict(
    dades_3d[
        ['Preu_Mitja_Actual', 'Creixement Històric (%)', 'Volatilitat_Std']
    ]
)

dades_3d['Estat_Mercat'] = dades_3d['Es_Anomalia'].apply(
    lambda x: 'Risc/Anomalia' if x == -1 else 'Mercat Normal'
)

grafic_3d = px.scatter_3d(
    dades_3d,
    x='Preu_Mitja_Actual',
    y='Creixement Històric (%)',
    z='Volatilitat_Std',
    color='Estat_Mercat',
    size='Total_Contractes',
    hover_name='Nom territori',
    color_discrete_map={'Risc/Anomalia': 'red', 'Mercat Normal': 'blue'},
    title='Gràfic 3D d\'Anomalies i Riscos Immobiliaris'
)

grafic_3d.show()


Els municipis més anòmals són:

In [ ]:
taula_anomalies = dades_3d[dades_3d['Estat_Mercat'] == 'Risc/Anomalia']

columnes_clau = [
    'Nom territori',
    'Preu_Mitja_Actual',
    'Creixement Històric (%)',
    'Volatilitat_Std'
]

llista_cignes_negres = taula_anomalies[columnes_clau]

llista_cignes_negres = llista_cignes_negres.sort_values(
    by='Volatilitat_Std', ascending=False
)

llista_cignes_negres = llista_cignes_negres.reset_index(drop=True)

llista_cignes_negres

Podem deduir que una bona estratègia d'inversió seria invertir en els municipis de la segmentació escollida, treient-li els que siguin dins la regió més anòmala. En el nostre cas, això ens resulta en la següent llista de municipis:

In [ ]:
pobles_perillosos = dades_3d[
    dades_3d['Estat_Mercat'] == 'Risc/Anomalia'
]['Nom territori']

cartera_segura = oportunitats_inversio[
    ~oportunitats_inversio['Nom territori'].isin(pobles_perillosos)
]

columnes_inversio = [
    'Nom territori',
    'Preu_Mitja_Actual',
    'Creixement Històric (%)',
    'Total_Contractes'
]

cartera_definitiva = (
    cartera_segura[columnes_inversio]
    .sort_values(by='Creixement Històric (%)', ascending=False)
    .reset_index(drop=True)
)

cartera_definitiva

En creuar la llista de municipis del clúster d'inversió (grup 2) amb el 5% d'anomalies de l'Isolation Forest, veiem que només descartem 3 municipis.Això representa una validació creuada dels nostres models: demostra que la segmentació prèvia amb K-Means ha estat altament efectiva aïllant l'exposició als mercats extrems.


# Conclusions

1. Predicció: Hem vist com els models polinòmics queien en l'overfitting. La regressió lineal aconseguia un error en la predicció decent donades les poques dades que hem emprat. Finalment el model en el que hem incorporat variables de retard ha aconseguit reduir l'error de la regressió lineal, però hem de ser conscients que no és útil per a prediccions a llarg plaç.

2. Segmentació: El K-Means m'ha permès aïllar el segment amb una barrera d'entrada assequible però amb una forta inèrcia de creixement històric.

3. Detecció d'anomalies: El fet que l'Isolation Forest global no hagi descartat quasi cap municipi del segment 2 ens confirma matemàticament que el K-Means ha fet bé la seva feina i ha deixat un clúster gairebé net de "bombolles" i volatilitats extremes.

# Possibles següents passos:

1. Afegir dades: Introduir variables com l'evolució de l'Euribor o l'IPC per rebaixar l'error del model de predicció. Podríem també experimentar amb models més complexos.

2. Buscar anomalies locals: Entrenar un nou Isolation Forest exclusivament amb els pobles del segment 2 per afinar encara més i detectar quins municipis estan sobrevalorats en comparació amb els seus iguals (anomalies locals).